# 01 - Database Exploration

Bu notebook'ta SQLite veritabanı incelenir, gerekli tablolar birleştirilir ve modelleme için ana veri seti hazırlanır.


In [14]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path.cwd()

# VS Code bazen çalışma klasörünü notebooks/ yapabilir. Buna karşı güvenli kontrol:
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "CompSciencePub.sqlite"

print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

conn = sqlite3.connect(DB_PATH)


Project root: /Users/alp/dev/journal-finder-project
Database path: /Users/alp/dev/journal-finder-project/data/CompSciencePub.sqlite
Database exists: True


In [15]:
tables = pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table'
ORDER BY name
""", conn)

tables


,name
0,AcademicContributor
1,AcademicFundGrand
2,AcademicHeading
3,AcademicKeyword
4,AcademicKeywordPlus
5,AcademicOrganization
6,AcademicOrganizationSet
7,AcademicRecord
8,AcademicRecordAbstract
9,AcademicRecordContributor


In [16]:
row_counts = []

for table in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS count FROM '{table}'", conn)["count"].iloc[0]
    row_counts.append({"table": table, "row_count": count})

row_counts_df = pd.DataFrame(row_counts).sort_values("row_count", ascending=False)
row_counts_df


,table,row_count
10,AcademicRecordContributorOrganizationSet,138860
14,AcademicRecordKeyword,101769
17,AcademicRecordSubject,86033
9,AcademicRecordContributor,84865
15,AcademicRecordKeywordPlus,83797
0,AcademicContributor,76343
11,AcademicRecordContributorSubOrganizationSet,62980
3,AcademicKeyword,52058
16,AcademicRecordSubHeading,27340
13,AcademicRecordHeading,24819


## Main Dataset Query

Bu sorgu şu bilgileri birleştirir:

- Makale bilgileri
- Abstract metni
- Dergi adı
- Document type
- Language
- Author keywords
- Web of Science Keyword Plus
- Web of Science subject/topic bilgileri


In [17]:
query = """
WITH
kw AS (
    SELECT 
        ark.AcademicRecordId,
        GROUP_CONCAT(ak.Name, '; ') AS author_keywords
    FROM AcademicRecordKeyword ark
    JOIN AcademicKeyword ak 
        ON ak.AcademicKeywordID = ark.AcademicKeywordId
    GROUP BY ark.AcademicRecordId
),
kwp AS (
    SELECT 
        arkwp.AcademicRecordId,
        GROUP_CONCAT(akp.Name, '; ') AS keyword_plus
    FROM AcademicRecordKeywordPlus arkwp
    JOIN AcademicKeywordPlus akp 
        ON akp.AcademicKeywordPlusID = arkwp.AcademicKeywordPlusId
    GROUP BY arkwp.AcademicRecordId
),
subj AS (
    SELECT 
        ars.AcademicRecordId,
        GROUP_CONCAT(asub.NameEn, '; ') AS subjects
    FROM AcademicRecordSubject ars
    JOIN AcademicSubject asub 
        ON asub.AcademicSubjectID = ars.AcademicSubjectId
    GROUP BY ars.AcademicRecordId
)
SELECT
    ar.AcademicRecordID AS academic_record_id,
    ar.WosUID AS wos_uid,
    ar.Title AS title,
    aa.AbstractText AS abstract_text,

    p.PublicationID AS publication_id,
    p.Name AS journal_name,
    p.Abbreviation AS journal_abbreviation,
    p.ISSN AS issn,

    ar.PubYear AS pub_year,
    ar.CiteCount AS cite_count,
    ar.ImpactFactor AS record_impact_factor,
    ar.QValue AS q_value,

    dt.NameEn AS document_type,
    l.Name AS language,
    pt.NameEn AS publication_type,

    kw.author_keywords,
    kwp.keyword_plus,
    subj.subjects

FROM AcademicRecord ar
JOIN AcademicRecordAbstract aa 
    ON aa.AcademicRecordId = ar.AcademicRecordID
JOIN Publication p 
    ON p.PublicationID = ar.PublicationId
LEFT JOIN DocumentType dt 
    ON dt.DocumentTypeID = ar.DocumentTypeId
LEFT JOIN Language l 
    ON l.LanguageID = ar.LanguageId
LEFT JOIN PublicationType pt 
    ON pt.PublicationTypeID = ar.PublicationTypeId
LEFT JOIN kw 
    ON kw.AcademicRecordId = ar.AcademicRecordID
LEFT JOIN kwp 
    ON kwp.AcademicRecordId = ar.AcademicRecordID
LEFT JOIN subj 
    ON subj.AcademicRecordId = ar.AcademicRecordID
WHERE aa.AbstractText IS NOT NULL
"""

df = pd.read_sql(query, conn)
df.head()


,academic_record_id,wos_uid,title,abstract_text,publication_id,journal_name,journal_abbreviation,issn,pub_year,cite_count,record_impact_factor,q_value,document_type,language,publication_type,author_keywords,keyword_plus,subjects
0,88652,WOS:000166979500001,An updated survey of GA-based multiobjective optimization techniques,"<p>After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a pop...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,337,None,None,Review,English,Journal,algorithms; artificial intelligence; genetic algorithms; multicriteria optimization; multiobjective optimization; vector optimization,GENETIC ALGORITHM; MULTICRITERIA OPTIMIZATION; STRUCTURAL OPTIMIZATION; ENGINEERING DESIGN; SEARCH; SOLVE,"Computer Science, Theory & Methods; Computer Science"
1,88653,WOS:000168229900003,The state of the art in distributed query processing,"<p>Distributed data processing is becoming a reality. Businesses want to do it for many reasons, and they often must do it in order to stay competitive. While much of the infrastructure for distri...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,298,None,None,Review,English,Journal,query optimization; query execution; client-server databases; middleware; multitier architectures; database application systems; wrappers; replication; caching; economic models for query processin...,DATABASE-SYSTEMS; DATA REPLICATION; PERFORMANCE; JOIN; OPTIMIZATION; ALGORITHMS; ACCESS,"Computer Science, Theory & Methods; Computer Science"
2,88654,WOS:000168229900001,Logical models of argument,<p>Logical models of argument formalize commonsense reasoning while taking process and computation seriously. This survey discusses the main ideas that characterize different logical models of arg...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,238,None,None,Review,English,Journal,defeasible argumentation; argumentative systems; defeasible reasoning,IMPLEMENTATION; FRAMEWORK,"Computer Science, Theory & Methods; Computer Science"
3,88655,WOS:000166979500002,Information retrieval on the Web,<p>In this paper we review studies of the growth of the Internet and technologies that are useful for information search and retrieval on the Web. We present data an the Internet from several diff...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,220,None,None,Review,English,Journal,algorithms; theory; clustering; indexing; information retrieval; Internet; knowledge management; search engine; World Wide Web,WORLD-WIDE-WEB; SEARCH; SYSTEM; DIRECTIONS; ENGINE; QUERY,"Computer Science, Theory & Methods; Computer Science"
4,88656,WOS:000168987700002,A guided tour to approximate string matching,<p>We survey the current techniques to cope with the problem of string matching that allows errors. This is becoming a more and more relevant issue for many fast growing areas such as information ...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2001,858,None,None,Review,English,Journal,algorithms; edit distance; Levenshtein distance; online string matching; text searching allowing errors,FAST ALGORITHMS; SUBQUADRATIC ALGORITHM; COMMON ANCESTORS; TEXT; MISMATCHES; SEARCH; CONSTRUCTION; HYPERTEXT; DISTANCES,"Computer Science, Theory & Methods; Computer Science"


In [18]:
print("Dataset shape:", df.shape)
print("Unique journals:", df["journal_name"].nunique())
print("Unique publications:", df["publication_id"].nunique())

df.info()


Dataset shape: (23061, 18)
Unique journals: 455
Unique publications: 455
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23061 entries, 0 to 23060
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   academic_record_id    23061 non-null  int64 
 1   wos_uid               23061 non-null  object
 2   title                 23061 non-null  object
 3   abstract_text         23061 non-null  object
 4   publication_id        23061 non-null  int64 
 5   journal_name          23061 non-null  object
 6   journal_abbreviation  23061 non-null  object
 7   issn                  23060 non-null  object
 8   pub_year              23061 non-null  int64 
 9   cite_count            23061 non-null  int64 
 10  record_impact_factor  0 non-null      object
 11  q_value               0 non-null      object
 12  document_type         23061 non-null  object
 13  language              23055 non-null  object
 14  publication_t

In [19]:
df[[
    "academic_record_id",
    "title",
    "journal_name",
    "pub_year",
    "document_type",
    "language",
    "publication_type",
    "author_keywords",
    "keyword_plus",
    "subjects",
    "abstract_text"
]].sample(5, random_state=42)


,academic_record_id,title,journal_name,pub_year,document_type,language,publication_type,author_keywords,keyword_plus,subjects,abstract_text
247,88900,"Additive Spanners and (alpha, beta)-Spanners",ACM TRANSACTIONS ON ALGORITHMS,2010,Article,English,Journal,Spanner; metric embedding,APPROXIMATE DISTANCE ORACLES; SHORTEST PATHS; WEIGHTED GRAPHS; SPARSE SPANNERS; ALL-PAIRS; ALGORITHMS; STRETCH; METRICS; TIME,"Computer Science, Theory & Methods; Mathematics, Applied; Computer Science; Mathematics","<p>An (alpha, beta)-spanner of an unweighted graph G is a subgraph H that distorts distances in G up to a multiplicative factor of a and an additive term beta. It is well known that any graph cont..."
12037,101296,Using fuzzy cognitive maps to identify multiple causes in troubleshooting systems,INTEGRATED COMPUTER-AIDED ENGINEERING,2008,Article,English,Journal,None,FAILURE MODES,"Computer Science, Artificial Intelligence; Computer Science, Interdisciplinary Applications; Engineering, Multidisciplinary; Computer Science; Engineering",<p>Fuzzy cognitive maps are a qualitative tool that can capture the extensive cause/effect relationships that an expert believes exist within a complicated system such as an electronic circuit. In...
1580,90240,"RAIDShield: Characterizing, Monitoring, and Proactively Protecting Against Disk Failures",ACM TRANSACTIONS ON STORAGE,2015,Article,English,Journal,reliability; experimentation; Magnetic disks; RAID; reliability; disk errors,None,"Computer Science, Hardware & Architecture; Computer Science, Software Engineering; Computer Science","<p>Modern storage systems orchestrate a group of disks to achieve their performance and reliability goals. Even though such systems are designed to withstand the failure of individual disks, failu..."
3128,91841,A Novel Picture Fuzzy Linguistic Aggregation Operator and Its Application to Group Decision-making,COGNITIVE COMPUTATION,2018,Article,English,Journal,Picture fuzzy set; Triangular norms and conorms; Multiple attribute group decision-making,ARCHIMEDEAN T-CONORM; SETS; VARIABLES; ALGORITHM; NUMBERS; SYSTEM,"Computer Science, Artificial Intelligence; Neurosciences; Computer Science; Neurosciences & Neurology","<p>The intuitionistic fuzzy set (IFS) is an effective tool to express uncertain and incomplete cognitive information with the membership, non-membership, and hesitance degrees. The picture fuzzy s..."
17249,106562,"On Lattices, Learning with Errors, Random Linear Codes, and Cryptography",JOURNAL OF THE ACM,2009,Article,English,Journal,algorithms; theory; lattice; Cryptography; Quantum computation; public key encryption; average-case hardness,REDUCTION; HARDNESS,"Computer Science, Hardware & Architecture; Computer Science, Information Systems; Computer Science, Software Engineering; Computer Science, Theory & Methods; Computer Science","<p>Our main result is a reduction from worst-case lattice problems such as GAPSVP and SIVP to a certain learning problem. This learning problem is a natural extension of the ""learning from parity ..."


## Basic Statistics

In [20]:
df["abstract_word_count"] = df["abstract_text"].astype(str).str.split().str.len()

df[["abstract_word_count", "cite_count", "record_impact_factor", "q_value"]].describe()


,abstract_word_count,cite_count
count,23061.000000,23061.000000
mean,164.852825,102.117775
std,63.121056,329.203161
min,13.000000,0.000000
25%,121.000000,15.000000
50%,159.000000,37.000000
75%,202.000000,92.000000
max,865.000000,14187.000000


In [21]:
df["journal_name"].value_counts().head(20)


journal_name
ACM COMPUTING SURVEYS                                     76
JOURNAL OF COMPUTER SCIENCE AND TECHNOLOGY                76
COMPUTER METHODS AND PROGRAMS IN BIOMEDICINE              76
COMPUTER JOURNAL                                          76
COMPUTER GRAPHICS FORUM                                   76
JOURNAL OF COMMUNICATIONS TECHNOLOGY AND ELECTRONICS      76
COMPUTER COMMUNICATIONS                                   76
COMPUTER APPLICATIONS IN ENGINEERING EDUCATION            76
JOURNAL OF COMPUTER AND SYSTEM SCIENCES                   76
COMPUTER AIDED GEOMETRIC DESIGN                           76
COMPUTER-AIDED DESIGN                                     76
IEEE JOURNAL ON SELECTED AREAS IN COMMUNICATIONS          76
COMPUTATIONAL GEOSCIENCES                                 76
INFORMATION AND COMPUTATION                               76
JOURNAL OF COMPUTER AND SYSTEMS SCIENCES INTERNATIONAL    76
JOURNAL OF COMPUTER INFORMATION SYSTEMS                   76
CMES-COMPUT

In [22]:
df["document_type"].value_counts(dropna=False)


document_type
Article               21982
Review                  979
Editorial Material       67
Letter                   13
Proceedings Paper         8
Reprint                   3
Software Review           3
Biographical-Item         2
Bibliography              2
Correction                1
Database Review           1
Name: count, dtype: int64

In [23]:
df["language"].value_counts(dropna=False)


language
English    23055
None           6
Name: count, dtype: int64

In [24]:
df["subjects"].value_counts(dropna=False).head(20)


subjects
Computer Science, Software Engineering; Computer Science                                                                                                                          1963
Computer Science, Artificial Intelligence; Computer Science                                                                                                                       1542
Computer Science, Information Systems; Computer Science                                                                                                                           1171
Computer Science, Theory & Methods; Computer Science                                                                                                                              1005
Computer Science, Information Systems; Telecommunications; Computer Science; Telecommunications                                                                                    956
Computer Science, Information Systems; Computer Science, Software Engineerin

## Clean Dataset

Model için ilk temizlik:

- Abstract boş olmayacak
- Journal boş olmayacak
- Abstract çok kısa olmayacak


In [25]:
df_clean = df.copy()

df_clean = df_clean.dropna(subset=["abstract_text", "journal_name"])
df_clean["abstract_text"] = df_clean["abstract_text"].astype(str).str.strip()
df_clean = df_clean[df_clean["abstract_text"].str.len() > 0]
df_clean = df_clean[df_clean["abstract_word_count"] >= 50]

print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)
print("Unique journals after cleaning:", df_clean["journal_name"].nunique())

df_clean.head()


Original shape: (23061, 19)
Clean shape: (22655, 19)
Unique journals after cleaning: 454


,academic_record_id,wos_uid,title,abstract_text,publication_id,journal_name,journal_abbreviation,issn,pub_year,cite_count,record_impact_factor,q_value,document_type,language,publication_type,author_keywords,keyword_plus,subjects,abstract_word_count
0,88652,WOS:000166979500001,An updated survey of GA-based multiobjective optimization techniques,"<p>After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a pop...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,337,None,None,Review,English,Journal,algorithms; artificial intelligence; genetic algorithms; multicriteria optimization; multiobjective optimization; vector optimization,GENETIC ALGORITHM; MULTICRITERIA OPTIMIZATION; STRUCTURAL OPTIMIZATION; ENGINEERING DESIGN; SEARCH; SOLVE,"Computer Science, Theory & Methods; Computer Science",153
1,88653,WOS:000168229900003,The state of the art in distributed query processing,"<p>Distributed data processing is becoming a reality. Businesses want to do it for many reasons, and they often must do it in order to stay competitive. While much of the infrastructure for distri...",11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,298,None,None,Review,English,Journal,query optimization; query execution; client-server databases; middleware; multitier architectures; database application systems; wrappers; replication; caching; economic models for query processin...,DATABASE-SYSTEMS; DATA REPLICATION; PERFORMANCE; JOIN; OPTIMIZATION; ALGORITHMS; ACCESS,"Computer Science, Theory & Methods; Computer Science",219
2,88654,WOS:000168229900001,Logical models of argument,<p>Logical models of argument formalize commonsense reasoning while taking process and computation seriously. This survey discusses the main ideas that characterize different logical models of arg...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,238,None,None,Review,English,Journal,defeasible argumentation; argumentative systems; defeasible reasoning,IMPLEMENTATION; FRAMEWORK,"Computer Science, Theory & Methods; Computer Science",100
3,88655,WOS:000166979500002,Information retrieval on the Web,<p>In this paper we review studies of the growth of the Internet and technologies that are useful for information search and retrieval on the Web. We present data an the Internet from several diff...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2000,220,None,None,Review,English,Journal,algorithms; theory; clustering; indexing; information retrieval; Internet; knowledge management; search engine; World Wide Web,WORLD-WIDE-WEB; SEARCH; SYSTEM; DIRECTIONS; ENGINE; QUERY,"Computer Science, Theory & Methods; Computer Science",160
4,88656,WOS:000168987700002,A guided tour to approximate string matching,<p>We survey the current techniques to cope with the problem of string matching that allows errors. This is becoming a more and more relevant issue for many fast growing areas such as information ...,11050,ACM COMPUTING SURVEYS,ACM COMPUT SURV,0360-0300,2001,858,None,None,Review,English,Journal,algorithms; edit distance; Levenshtein distance; online string matching; text searching allowing errors,FAST ALGORITHMS; SUBQUADRATIC ALGORITHM; COMMON ANCESTORS; TEXT; MISMATCHES; SEARCH; CONSTRUCTION; HYPERTEXT; DISTANCES,"Computer Science, Theory & Methods; Computer Science",103


In [26]:
output_path = PROJECT_ROOT / "data" / "master_articles.csv"
df_clean.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: /Users/alp/dev/journal-finder-project/data/master_articles.csv
